In [ ]:
import os
import glob
import pandas as pd

# Define relative path from 'master_datasets' to 'data_log/data_jerry'
# '..' steps up to the project root 'DEP_Proj'
# Correcting the path to the current directory as the files were found in /content/
data_dir = "."

# Match all cleaned CSV files
csv_files = glob.glob(os.path.join(data_dir, "*_clean*.csv"))

# Store dataframes
df_list = []

for file in csv_files:
    df = pd.read_csv(file)
    # Store source filename to track data provenance
    df["source_file"] = os.path.basename(file)
    df_list.append(df)

# Combine all DataFrames vertically
master_df = pd.concat(df_list, ignore_index=True)

# Save the master dataset in the current directory (master_datasets)
output_path = "jerry_master.csv"
master_df.to_csv(output_path, index=False)

print(f"Successfully combined {len(csv_files)} files into '{output_path}'.")
print(f"Master Dataset Shape: {master_df.shape}")

Successfully combined 2 files into 'jerry_master.csv'.
Master Dataset Shape: (7098, 10)


### Resolving `ValueError: No objects to concatenate`

The previous cell failed because no CSV files were found. This usually means:

1.  **The files are not in the expected location:** The `data_dir` variable is set to `../data_log/data_jerry`. In Google Colab, the working directory is usually `/content/`. If your files are located differently, you'll need to adjust this path.
2.  **The files haven't been uploaded:** If the `*_clean*.csv` files are on your local machine, you need to upload them to the Colab environment first.

To troubleshoot, you can use the following code cell to either upload files or list the contents of a directory to confirm their presence and exact names.

In [ ]:
import os

# --- Option 1: Upload files from your local machine ---
# If your files are on your local computer, uncomment and run the following line.
# This will open a file selection dialog.
# from google.colab import files
# uploaded = files.upload()

# After uploading, you might need to move them to the expected directory.
# For example, if you want them in `../data_log/data_jerry` (relative to current script location).
# If your files are directly in `/content/`, you might need to create the `data_log/data_jerry` structure.
# !mkdir -p ../data_log/data_jerry # Creates the directory if it doesn't exist
# !mv *.csv ../data_log/data_jerry/ # Moves all uploaded CSVs to the target directory

# --- Option 2: Verify existing files in Colab ---
# If you believe the files are already in Colab, uncomment and adjust the path below
# to list the contents of your expected directory.

# This lists files in the directory your script tried to access:
expected_dir_for_check = os.path.join("../", "data_log", "data_jerry")
print(f"Checking directory: {expected_dir_for_check}")

# Ensure the directory exists before trying to list its contents
if os.path.exists(expected_dir_for_check):
    print("Files in directory:")
    for root, dirs, files in os.walk(expected_dir_for_check):
        for file in files:
            if file.endswith('_clean.csv'): # Filter for the expected files
                print(os.path.join(root, file))
else:
    print(f"Directory does not exist: {expected_dir_for_check}")
    print("Please create the directory or ensure the `data_dir` path in the previous cell is correct.")


# After ensuring files are present, you can rerun the first cell (`e4581318`).

Checking directory: ../data_log/data_jerry
Directory does not exist: ../data_log/data_jerry
Please create the directory or ensure the `data_dir` path in the previous cell is correct.


data cleaning, inspect structure

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = "jerry_master.csv"
df = pd.read_csv(file_path)

# Display basic information
print("--- Initial Overview ---")
print(f"Dataset Shape: {df.shape}")
print("\n--- Data Types & Missing Values ---")
print(df.info())
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

df.head()

--- Initial Overview ---
Dataset Shape: (7098, 10)

--- Data Types & Missing Values ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7098 entries, 0 to 7097
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Sample       7098 non-null   int64  
 1   Timestamp    7098 non-null   object 
 2   Elapsed_s    7098 non-null   float64
 3   AccX         7098 non-null   float64
 4   AccY         7098 non-null   float64
 5   AccZ         7098 non-null   float64
 6   GyroX        7098 non-null   float64
 7   GyroY        7098 non-null   float64
 8   GyroZ        7098 non-null   float64
 9   source_file  7098 non-null   object 
dtypes: float64(7), int64(1), object(2)
memory usage: 554.7+ KB
None

--- Missing Values Count ---
Sample         0
Timestamp      0
Elapsed_s      0
AccX           0
AccY           0
AccZ           0
GyroX          0
GyroY          0
GyroZ          0
source_file    0
dtype: int64


,Sample,Timestamp,Elapsed_s,AccX,AccY,AccZ,GyroX,GyroY,GyroZ,source_file
0,0,2026-08-03 13:23:11.086,0.000,-0.11,-0.11,0.79,42.79,71.47,-122.44,imu_log_20260803_132305_clean.csv
1,1,2026-08-03 13:23:11.116,0.030,-0.27,-0.02,0.88,136.60,131.71,-76.42,imu_log_20260803_132305_clean.csv
2,2,2026-08-03 13:23:11.145,0.059,-0.40,0.06,1.08,175.54,132.93,-41.08,imu_log_20260803_132305_clean.csv
3,3,2026-08-03 13:23:11.160,0.074,-0.40,0.06,1.08,175.54,132.93,-41.08,imu_log_20260803_132305_clean.csv
4,4,2026-08-03 13:23:11.189,0.103,-0.46,0.13,1.11,96.68,62.56,-29.97,imu_log_20260803_132305_clean.csv


In [ ]:
# 1. Remove duplicate rows (ignoring pure identical duplicates)
initial_rows = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_rows - len(df)} duplicate rows.")

# 2. Handle missing values
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].ffill().bfill()

# Check remaining missing values
print(f"Remaining nulls: {df.isnull().sum().sum()}")

Removed 0 duplicate rows.
Remaining nulls: 0


In [ ]:
# Assuming there is a timestamp column (e.g., 'timestamp', 'time', or 'date')
timestamp_col = [col for col in df.columns if 'time' in col.lower() or 'date' in col.lower()]

if timestamp_col:
    col_name = timestamp_col[0]
    df[col_name] = pd.to_datetime(df[col_name])
    df = df.sort_values(by=col_name).reset_index(drop=True)
    print(f"Parsed and sorted by timestamp column: '{col_name}'")
else:
    print("No timestamp column detected; skipping time sorting.")

Parsed and sorted by timestamp column: 'Timestamp'


In [ ]:
# Identify numeric columns excluding metadata like source_file
features = df.select_dtypes(include=[np.number]).columns

for col in features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Clip extreme values to the upper and lower threshold limits
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

print("Outlier clipping complete using IQR method.")

Outlier clipping complete using IQR method.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Select feature columns to scale
feature_cols = df.select_dtypes(include=[np.number]).columns

df_scaled = df.copy()
df_scaled[feature_cols] = scaler.fit_transform(df[feature_cols])

print("Features successfully scaled.")
df_scaled.head()

Features successfully scaled.


,Sample,Timestamp,Elapsed_s,AccX,AccY,AccZ,GyroX,GyroY,GyroZ,source_file
0,-1.730820,2026-08-03 13:20:26.620,-1.677397,-0.524125,-0.829643,-0.009632,0.072873,-0.099566,-0.062667,imu_log_20260803_132017_clean.csv
1,-1.729845,2026-08-03 13:20:26.621,-1.677363,0.452415,2.042085,1.036044,0.883252,-0.047548,0.796697,imu_log_20260803_132017_clean.csv
2,-1.728870,2026-08-03 13:20:26.621,-1.677363,-0.686882,1.702699,0.936456,-0.984850,0.051609,-0.388191,imu_log_20260803_132017_clean.csv
3,-1.727894,2026-08-03 13:20:26.622,-1.677329,-0.686882,1.702699,0.936456,-0.984850,0.051609,-0.388191,imu_log_20260803_132017_clean.csv
4,-1.726919,2026-08-03 13:20:26.623,-1.677295,-0.469873,0.475688,0.438515,-2.173204,0.211725,-1.273308,imu_log_20260803_132017_clean.csv


In [ ]:
output_cleaned_path = "jerry_master_preprocessed.csv"
df_scaled.to_csv(output_cleaned_path, index=False)

print(f"Preprocessed dataset successfully saved to: {output_cleaned_path}")
print(f"Final Processed Shape: {df_scaled.shape}")

Preprocessed dataset successfully saved to: jerry_master_preprocessed.csv
Final Processed Shape: (7098, 10)
